In [1]:
from dataclasses import replace
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

import xarray as xr

# Also allow execution from the documentation's usage directory.
repository = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "examples" / "multisector_handoffs.py").is_file()
)
sys.path.insert(0, str(repository))

from examples.multisector_handoffs import model_options, synthetic_inputs
from examples.rhime_customisation.likelihoods import likelihood_builder
from openghg_inversions.postprocessing.make_outputs import make_sector_flux_outputs
from openghg_inversions.rhime import multisector_stages as handoffs
from openghg_inversions.rhime.multisector import run_multisector
from openghg_inversions.rhime.params import DataConfig, RhimeConfig
from openghg_inversions.rhime.sampling import RhimeSampler
from openghg_inversions.rhime.specs import RhimeOutputSpec

workspace = TemporaryDirectory(prefix="multisector-handoffs-")
destination = Path(workspace.name)
merged, options, flux, responses, observations = synthetic_inputs(destination)
{"acquired sources": list(merged.flux_data), "sectors": model_options(0.2)["sector_sources"]}

{'acquired sources': ['inventory', 'supplemental'],
 'sectors': {'biosphere': 'supplemental', 'fossil': 'inventory'}}

In [2]:
data = DataConfig.from_params(options, multisector=True)
preparation_manifest = handoffs.prepare(
    data,
    merged_data=merged,
    output_dir=destination / "prepared",
)
prepared, recorded_data, generation = handoffs.load_preparation(preparation_manifest)
prepared.inv_inputs[["mf", "mf_error", "H"]].compute()

TIMING rhime.prepare_inputs.obs_filtering seconds=0.000005 maxrss_kb=674880 sites=1 filters=false
Computing basis took 0.02077460289001465s.
TIMING rhime.prepare_inputs.basis_build seconds=0.020828 maxrss_kb=674880 basis_algorithm=weighted


TIMING rhime.prepare_inputs.footprint_sensitivity seconds=0.380197 maxrss_kb=674880 site=TAC nmeasure=4 state_size=1 sources=2
TIMING rhime.prepare_inputs.footprint_sensitivity_total seconds=0.380434 maxrss_kb=674880 sites=1
TIMING rhime.prepare_inputs.make_inv_inputs seconds=0.026821 maxrss_kb=674880 sites=1
TIMING rhime.prepare_inputs.prepared_dims seconds=0.000000 maxrss_kb=674880 nmeasure=4 sites=1 sources=2 basis_source=datatree
TIMING rhime.prepare_inputs seconds=0.453131 maxrss_kb=674880 multisector=true nmeasure=4 sites=1 sources=2 basis_source=datatree


<xarray.Dataset> Size: 344B
Dimensions:   (nmeasure: 4, state: 1, source: 2)
Coordinates:
  * nmeasure  (nmeasure) object 32B MultiIndex
  * site      (nmeasure) <U3 48B 'TAC' 'TAC' 'TAC' 'TAC'
  * time      (nmeasure) datetime64[ns] 32B 2019-01-02 2019-01-03 ... 2019-01-05
  * state     (state) int64 8B 0
  * source    (source) <U12 96B 'inventory' 'supplemental'
Data variables:
    mf        (nmeasure) float64 32B 1.3 1.15 1.28 1.4
    mf_error  (nmeasure) float64 32B 0.1 0.1 0.1 0.1
    H         (state, nmeasure, source) float64 64B 1.0 0.2 0.5 ... 0.4 1.2 0.1

In [3]:
sampler = RhimeSampler(
    draws=8,
    tune=8,
    chains=1,
    sample_kwargs={"cores": 1, "random_seed": 42},
    sample_prior_predictive=False,
    sample_posterior_predictive=False,
)
normal_model = model_options(0.2)
normal_manifest = handoffs.sample(
    preparation_manifest,
    model=normal_model,
    sampler=sampler,
    output_dir=destination / "normal",
)

TIMING rhime.model_inputs_materialize seconds=0.001322 maxrss_kb=674880 variables=H,mf,mf_error coordinates=


Only 8 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


Initializing NUTS using jitter+adapt_diag...


TIMING rhime.model_build seconds=0.071322 maxrss_kb=674880 multisector=true


Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


/home/runner/work/openghg_inversions/openghg_inversions/source/.tox/docs-full/lib/python3.12/site-packages/pymc/sampling/mcmc.py:1178: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 1 chain for 8 tune and 8 draw iterations (8 + 8 draws total) took 0 seconds.


TIMING rhime.sampler.pm_sample seconds=12.975381 maxrss_kb=674880 draws=8 tune=8 chains=1 nuts_sampler=pymc
TIMING rhime.sampler.sample_stats seconds=0.000000 maxrss_kb=674880 n_steps_mean=12.000000 n_steps_max=15.000000 tree_depth_mean=3.625000 tree_depth_max=4.000000 step_size_mean=0.024416 acceptance_rate_mean=0.968719 divergences=0
TIMING rhime.sampler.burn_slicing seconds=0.005137 maxrss_kb=674880 burn=0
TIMING rhime.sampler.coord_restore seconds=0.009475 maxrss_kb=674880 restored=true
TIMING rhime.sampler.convergence seconds=0.025306 maxrss_kb=674880 status=fail message=ArviZ_detected_sampling_problems._Assessment_incomplete:_non-finite_diagnostics_for_x_biosphere_active;_non-finite_diagnostics_for_x_fossil_active;_between-chain_convergence_requires_at_least_two_chains. assessed_variables=x_biosphere_active,x_fossil_active chains=1 draws_per_chain=8 arviz_version=1.3.0 arviz_has_errors=true divergences=0 unavailable=non-finite_diagnostics_for_x_biosphere_active,non-finite_diagnos

In [4]:
student_model = model_options(0.6, student_t=True)
student_manifest = handoffs.sample(
    preparation_manifest,
    model=student_model,
    sampler=sampler,
    likelihood_builder=likelihood_builder,
    output_dir=destination / "student",
)

Only 8 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


Initializing NUTS using jitter+adapt_diag...


TIMING rhime.model_inputs_materialize seconds=0.001252 maxrss_kb=674880 variables=H,mf,mf_error coordinates=
TIMING rhime.model_build seconds=0.021483 maxrss_kb=674880 multisector=true


Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


/home/runner/work/openghg_inversions/openghg_inversions/source/.tox/docs-full/lib/python3.12/site-packages/pymc/sampling/mcmc.py:1178: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 1 chain for 8 tune and 8 draw iterations (8 + 8 draws total) took 0 seconds.


TIMING rhime.sampler.pm_sample seconds=4.525277 maxrss_kb=691528 draws=8 tune=8 chains=1 nuts_sampler=pymc
TIMING rhime.sampler.sample_stats seconds=0.000000 maxrss_kb=691528 n_steps_mean=5.250000 n_steps_max=7.000000 tree_depth_mean=2.625000 tree_depth_max=3.000000 step_size_mean=0.047678 acceptance_rate_mean=0.862313 divergences=0
TIMING rhime.sampler.burn_slicing seconds=0.005082 maxrss_kb=691528 burn=0
TIMING rhime.sampler.coord_restore seconds=0.009375 maxrss_kb=691528 restored=true
TIMING rhime.sampler.convergence seconds=0.025132 maxrss_kb=691528 status=fail message=ArviZ_detected_sampling_problems._Assessment_incomplete:_non-finite_diagnostics_for_x_biosphere_active;_non-finite_diagnostics_for_x_fossil_active;_between-chain_convergence_requires_at_least_two_chains. assessed_variables=x_biosphere_active,x_fossil_active chains=1 draws_per_chain=8 arviz_version=1.3.0 arviz_has_errors=true divergences=0 unavailable=non-finite_diagnostics_for_x_biosphere_active,non-finite_diagnostic

In [5]:
normal = handoffs.postprocess(
    normal_manifest,
    output=RhimeOutputSpec(output_format="none"),
)
student = handoffs.postprocess(
    student_manifest,
    output=RhimeOutputSpec(output_format="none"),
)
exported = handoffs.postprocess(
    student_manifest,
    output=RhimeOutputSpec(
        output_format="inv_out",
        output_name="student-replay",
        output_path=str(destination / "exports"),
    ),
)
assert student.trace.equals(exported.trace)
student.run_metadata["start_date"], student.run_metadata["end_date"]

TIMING rhime.output.multisector_diagnostics seconds=0.163340 maxrss_kb=692860
TIMING rhime.output.multisector_diagnostics_netcdf_write seconds=0.006458 maxrss_kb=693884 path=/home/runner/work/openghg_inversions/openghg_inversions/source/.tox/docs-full/tmp/multisector-handoffs-dujspvai/exports/student-replay2019-01-01_sector_flux_diagnostics.nc
TIMING rhime.output.inversion_output_save seconds=0.059010 maxrss_kb=694396 path=/home/runner/work/openghg_inversions/openghg_inversions/source/.tox/docs-full/tmp/multisector-handoffs-dujspvai/exports/student-replay2019-01-01_inversion_output.nc


('2019-01-01', '2019-02-01')

In [6]:
normal_flux = make_sector_flux_outputs(
    normal,
    stats=["mean"],
    report_flux_on_inversion_grid=False,
)
student_flux = make_sector_flux_outputs(
    student,
    stats=["mean"],
    report_flux_on_inversion_grid=False,
)
xr.concat(
    [normal_flux, student_flux],
    dim=xr.IndexVariable("model", ["Normal", "Student-t"]),
)

<xarray.Dataset> Size: 432B
Dimensions:                           (model: 2, flux_time: 1, lat: 2, lon: 2)
Coordinates:
  * model                             (model) <U9 72B 'Normal' 'Student-t'
  * flux_time                         (flux_time) datetime64[ns] 8B 2019-01-01
  * lat                               (lat) float64 16B 50.0 51.0
  * lon                               (lon) float64 16B -2.0 -1.0
Data variables:
    flux_total_posterior_mean         (model, flux_time, lat, lon) float64 64B ...
    flux_biosphere_posterior_mean     (model, flux_time, lat, lon) float64 64B ...
    flux_fossil_posterior_mean        (model, flux_time, lat, lon) float64 64B ...
    scaling_biosphere_posterior_mean  (model, lat, lon) float64 64B 0.9288 .....
    scaling_fossil_posterior_mean     (model, lat, lon) float64 64B 1.083 ......
Attributes:
    openghg_inversions:non_finite_flux:  {"checked":"not_counted","context":"...
    basis_reconstruction_path:           operator-backed
    basis_artifact_source:               loaded-datatree
    basis_artifact_path:                 /home/runner/work/openghg_inversions...

In [7]:
config = RhimeConfig.from_params(
    {
        **options,
        **student_model,
        "output_name": "student-parity",
        "output_format": "none",
        "save_inversion_output": False,
    },
    multisector=True,
)
config = replace(config, sampler=sampler)
full = run_multisector(
    config=config,
    merged_data=merged,
    likelihood_builder=likelihood_builder,
)
resumed = run_multisector(
    config=config,
    prepared=prepared,
    likelihood_builder=likelihood_builder,
)
for result in (full, resumed):
    xr.testing.assert_allclose(
        student_flux,
        make_sector_flux_outputs(
            result,
            stats=["mean"],
            report_flux_on_inversion_grid=False,
        ),
    )
print("Full, prepared-input and replayed flux products agree.")

TIMING rhime.runner_setup seconds=0.000016 maxrss_kb=694396 multisector=true
TIMING rhime.prepare_inputs.obs_filtering seconds=0.000005 maxrss_kb=694396 sites=1 filters=false


Only 8 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


Initializing NUTS using jitter+adapt_diag...


Computing basis took 0.01234889030456543s.
TIMING rhime.prepare_inputs.basis_build seconds=0.012426 maxrss_kb=694908 basis_algorithm=weighted
TIMING rhime.prepare_inputs.footprint_sensitivity seconds=0.010261 maxrss_kb=694908 site=TAC nmeasure=4 state_size=1 sources=2
TIMING rhime.prepare_inputs.footprint_sensitivity_total seconds=0.010418 maxrss_kb=694908 sites=1
TIMING rhime.prepare_inputs.make_inv_inputs seconds=0.016508 maxrss_kb=694908 sites=1
TIMING rhime.prepare_inputs.prepared_dims seconds=0.000000 maxrss_kb=694908 nmeasure=4 sites=1 sources=2 basis_source=datatree
TIMING rhime.prepare_inputs seconds=0.063213 maxrss_kb=694908 multisector=true nmeasure=4 sites=1 sources=2 basis_source=datatree
TIMING rhime.model_inputs_materialize seconds=0.001280 maxrss_kb=694908 variables=H,mf,mf_error coordinates=
TIMING rhime.model_build seconds=0.021345 maxrss_kb=694908 multisector=true


Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


... (4 outputs left out)

Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


/home/runner/work/openghg_inversions/openghg_inversions/source/.tox/docs-full/lib/python3.12/site-packages/pymc/sampling/mcmc.py:1178: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 1 chain for 8 tune and 8 draw iterations (8 + 8 draws total) took 0 seconds.


TIMING rhime.sampler.pm_sample seconds=0.431847 maxrss_kb=705532 draws=8 tune=8 chains=1 nuts_sampler=pymc
TIMING rhime.sampler.sample_stats seconds=0.000000 maxrss_kb=705532 n_steps_mean=5.250000 n_steps_max=7.000000 tree_depth_mean=2.625000 tree_depth_max=3.000000 step_size_mean=0.047678 acceptance_rate_mean=0.862313 divergences=0
TIMING rhime.sampler.burn_slicing seconds=0.004927 maxrss_kb=705532 burn=0
TIMING rhime.sampler.coord_restore seconds=0.009368 maxrss_kb=705532 restored=true
TIMING rhime.sampler.convergence seconds=0.025177 maxrss_kb=705532 status=fail message=ArviZ_detected_sampling_problems._Assessment_incomplete:_non-finite_diagnostics_for_x_biosphere_active;_non-finite_diagnostics_for_x_fossil_active;_between-chain_convergence_requires_at_least_two_chains. assessed_variables=x_biosphere_active,x_fossil_active chains=1 draws_per_chain=8 arviz_version=1.3.0 arviz_has_errors=true divergences=0 unavailable=non-finite_diagnostics_for_x_biosphere_active,non-finite_diagnostic

In [8]:
workspace.cleanup()